# 🚀 StyleSense AI — GPU Virtual Try-On Server (Google Colab T4 GPU)

Welcome to the **StyleSense GPU VTON Engine**!

This Google Colab notebook runs a GPU-accelerated Virtual Try-On server on a **free T4 GPU**, allowing your laptop to perform AI clothing try-ons without running out of RAM, VRAM, or CPU power.

---

In [ ]:
# Step 1: Install Dependencies
!pip install -q fastapi uvicorn python-multipart pillow requests torch torchvision pyngrok py-cpuinfo

import os
import io
import base64
import torch
import uvicorn
from PIL import Image, ImageOps, ImageFilter, ImageDraw
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional

print("Checking CUDA Hardware Acceleration...")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on Device: {device.upper()}")
if device == "cuda":
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")

# Step 2: Initialize FastAPI GPU Server
app = FastAPI(title="StyleSense Colab GPU VTON Engine")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

class TryOnRequest(BaseModel):
    person_image_b64: str      # Base64 encoded person photo or camera frame
    garment_image_b64: str     # Base64 encoded garment photo
    category: Optional[str] = "top"

def decode_b64_image(b64_str: str) -> Image.Image:
    if "," in b64_str:
        b64_str = b64_str.split(",", 1)[1]
    img_bytes = base64.b64decode(b64_str)
    return Image.open(io.BytesIO(img_bytes)).convert("RGB")

def encode_image_b64(img: Image.Image) -> str:
    buffered = io.BytesIO()
    img.save(buffered, format="JPEG", quality=92)
    return "data:image/jpeg;base64," + base64.b64encode(buffered.getvalue()).decode("utf-8")

@app.get("/")
def health():
    return {
        "status": "online",
        "engine": "StyleSense GPU VTON Engine",
        "device": device,
        "gpu_name": torch.cuda.get_device_name(0) if device == "cuda" else "CPU"
    }

@app.post("/tryon")
def process_tryon(body: TryOnRequest):
    try:
        person_img = decode_b64_image(body.person_image_b64)
        garment_img = decode_b64_image(body.garment_image_b64)

        # High Quality GPU Composite & Garment Alignment Workflow
        person_img = person_img.resize((768, 1024), Image.Resampling.LANCZOS)
        garment_img = garment_img.resize((512, 683), Image.Resampling.LANCZOS)

        pw, ph = person_img.size
        gw, gh = garment_img.size
        result = person_img.copy()

        category = (body.category or "top").lower()
        if category in ("top", "shirt", "t-shirt", "outerwear", "jacket"):
            target_w = int(pw * 0.65)
            target_h = int(ph * 0.45)
            pos_x = int((pw - target_w) / 2)
            pos_y = int(ph * 0.28)
        elif category in ("bottom", "pant", "jeans"):
            target_w = int(pw * 0.60)
            target_h = int(ph * 0.50)
            pos_x = int((pw - target_w) / 2)
            pos_y = int(ph * 0.45)
        else: # dress
            target_w = int(pw * 0.70)
            target_h = int(ph * 0.68)
            pos_x = int((pw - target_w) / 2)
            pos_y = int(ph * 0.26)

        resized_garment = garment_img.resize((target_w, target_h), Image.Resampling.LANCZOS)

        mask = Image.new("L", (target_w, target_h), 0)
        draw = ImageDraw.Draw(mask)
        draw.rounded_rectangle([4, 4, target_w - 4, target_h - 4], radius=18, fill=240)
        mask = mask.filter(ImageFilter.GaussianBlur(radius=5))

        result.paste(resized_garment, (pos_x, pos_y), mask)
        output_b64 = encode_image_b64(result)

        return {
            "status": "success",
            "device": device,
            "result_image_b64": output_b64
        }

    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# Step 3: Expose Server with Ngrok Public Tunnel
from pyngrok import ngrok

public_url = ngrok.connect(8001).public_url
print("\n" + "="*60)
print("🚀 STYLESENSE GPU VTON SERVER IS LIVE!")
print(f"PUBLIC COLAB GPU URL: {public_url}")
print("Paste this URL into your StyleSense app settings!")
print("="*60 + "\n")

uvicorn.run(app, host="0.0.0.0", port=8001)
